# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

Below, we enumerate the record sets and their fields referenced by their `@id`s for clarity and reproducibility.

In [ ]:
# Collect record sets by @id
record_sets = []
field_map = dict()

croissant = dataset.metadata.to_json()

# Some Croissant schemas have 'recordSet' in metadata as a list of objects with @id and properties
if 'recordSet' in croissant and isinstance(croissant['recordSet'], list):
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in croissant['recordSet']]
    # Collect fields for each recordSet by their @id
    for rs in croissant['recordSet']:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        if isinstance(rs, dict) and 'field' in rs:
            fields = rs['field']
            field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]
            field_map[rs_id] = field_ids
else:
    # Attempt to get record sets from schema dataset
    # fallback: manual definitions using known structure from dataset description
    # The dataset has three main tables: Table 1, Table 2, Table 3; simulate their @ids
    record_sets = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3'
    ]
    field_map[record_sets[0]] = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_age',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_height',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_clitomegaly',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_hirsutism',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_alopecia',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_acne',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_oligomenorrhea',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table1_infertility'
    ]
    field_map[record_sets[1]] = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2_hormone',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2_ct',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table2_ultrasound'
    ]
    field_map[record_sets[2]] = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_chromosome',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_variant',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_population_frequency',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_zygosity',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_pathogenicity',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table3_phenotype'
    ]

print("Record Sets and their Fields (@id):")
for rs_id in record_sets:
    print(f"Record Set: {rs_id}")
    if rs_id in field_map:
        print("  Fields:")
        for field_id in field_map[rs_id]:
            print(f"    {field_id}")
    print()

In [ ]:
# Show exemplar records for each record set, referenced by @id
for rs_id in record_sets:
    print(f"Sample records for Record Set {rs_id}:")
    try:
        for i, x in enumerate(dataset.records(record_set=rs_id)):
            print(json.dumps(x, indent=2))
            if i >= 1:  # Print only first 2 records for brevity
                break
    except Exception as e:
        print(f"  Unable to load records: {e}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis. Each entity is referenced using its `@id`.

Proceeding, we collect all records for each record set and store them in a dictionary of DataFrames keyed by their record set `@id`.

In [ ]:
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id} with columns:")
            print(df.columns.tolist())
        else:
            print(f"No records returned for {record_set_id}")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Example: Display first few rows for Table 1
table1_id = record_sets[0]
if table1_id in dataframes:
    print("\nHead of Table 1:")
    display(dataframes[table1_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All operations are performed referencing columns using their `@id`.**

For demonstration, we'll use `Table 1` (clinical features) and analyze the `age` field (referenced by its `@id`). We'll filter records where age (at diagnosis) is above a certain threshold and normalize the age.

In [ ]:
# Select a numeric field for analysis (age)
table1_id = record_sets[0]
numeric_field = field_map[table1_id][0]  # 'table1_age' field @id

# Define threshold
threshold = 25

df = dataframes[table1_id]
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df)

    # Normalize the 'age' field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]])

    # Grouping by another field, such as presence of hirsutism (boolean)
    group_field = field_map[table1_id][3]  # 'table1_hirsutism' @id
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (Mean {numeric_field}):")
        display(grouped_df)
else:
    print(f"Field {numeric_field} not found in DataFrame columns.")

## 5. Visualization
Visualize distributions or relationships between fields using their `@id`s. For example, visualize the distribution of ages or the relationship between age and presence of hirsutism.

Below, we use matplotlib for quick visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age distribution (numeric_field by its @id)
if table1_id in dataframes:
    df = dataframes[table1_id]
    if numeric_field in df.columns:
        plt.figure(figsize=(6,3))
        sns.histplot(df[numeric_field], bins=5, kde=True)
        plt.xlabel(numeric_field)
        plt.title("Distribution of Age at Diagnosis")
        plt.show()

# Scatterplot: Age vs. Height (both referenced by their @id)
height_field = field_map[table1_id][1]
if all(f in df.columns for f in [numeric_field, height_field]):
    plt.figure(figsize=(5,4))
    plt.scatter(df[numeric_field], df[height_field])
    plt.xlabel(numeric_field)
    plt.ylabel(height_field)
    plt.title("Age vs. Height")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Used the Croissant schema (`mlcroissant`) to access a clinical dataset referencing all entities by their `@id`.
- Loaded three record sets (Table 1: clinical features, Table 2: hormone & imaging, Table 3: genotype).
- Performed basic data processing and normalization.
- Illustrated visualization of age distribution and age-height relationship using field `@id`s.
- Dataset is limited in scale but provides valuable insight for genotype–phenotype relationships in NC-21OHD-associated infertility; caution is advised for generalization.

For further exploration, consult the Croissant schema for deeper field definitions and relationships.